In [ ]:
import os
import argparse
import logging
from ast import literal_eval  # para evaluar cadenas que contienen listas literales

import numpy as np
import pandas as pd
import time
# Importamos utilidades de scikit-learn para preprocesamiento, reducción y clustering
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Configuración básica del logger para imprimir progreso
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

# Directorio donde guardamos salidas (crea si no existe)
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def extract_primary_genre(genres_field):
    """Extrae el primer género de una columna que puede ser:
    - NaN
    - una lista de Python (ej. ['Action','Adventure']) serializada como string
    - una cadena separada por comas ('Action, Adventure')

    Devuelve None (np.nan) si no hay género.
    """
    # Si es NaN devolvemos NaN
    if pd.isna(genres_field):
        return np.nan
    # Si ya es una lista Python en memoria, tomar el primer elemento
    if isinstance(genres_field, list):
        return genres_field[0] if len(genres_field) > 0 else np.nan
    # Convertir a string y limpiar espacios
    s = str(genres_field).strip()
    # Intentar interpretar cadenas que contienen una lista literal (p. ej. "['A','B']")
    try:
        parsed = literal_eval(s)
        if isinstance(parsed, (list, tuple)) and len(parsed) > 0:
            return parsed[0]
    except Exception:
        # Si falla, seguimos con el fallback
        pass
    # Si tiene comas, asumimos formato 'A, B, C' y tomamos el primero
    if ',' in s:
        return s.split(',')[0].strip().strip("'\"")
    # En cualquier otro caso devolvemos el string limpio
    return s.strip().strip("'\"")

def cluster_purity(labels_true, labels_pred):
    """Calcula la pureza de clusters (proporción de elementos correctamente asignados
    si usamos la etiqueta más frecuente de cada cluster).

    labels_true: array-like de etiquetas verdaderas (single-label por fila)
    labels_pred: etiquetas de cluster producidas por el algoritmo
    """
    # Creamos un DataFrame temporal para agrupar
    dfc = pd.DataFrame({'true': labels_true, 'pred': labels_pred})
    # Rellenar predictores nulos con -1 (ruido) para contar adecuadamente
    dfc['pred'] = dfc['pred'].fillna(-1)
    total = len(dfc)
    if total == 0:
        return np.nan
    pur_sum = 0
    # Para cada cluster, sumar el número de elementos de la clase mayoritaria
    for c in dfc['pred'].unique():
        sub = dfc[dfc['pred'] == c]
        if len(sub) == 0:
            continue
        top = sub['true'].value_counts().iloc[0]
        pur_sum += top
    # Dividir por el total para obtener la pureza global
    return pur_sum / total

In [ ]:
def load_and_merge(animes_path='animes.csv', dataset_path='dataset_completo.csv', ratings_path='ratings.csv'):
    """Carga `animes.csv` y `dataset_completo.csv`, los une y agrega estadísticas de `ratings.csv`.

    ratings.csv se procesa por chunks para no cargar todo en memoria cuando es grande.
    """
    # Cargar archivo de animes
    logging.info('Cargando animes desde %s', animes_path)
    animes = pd.read_csv(animes_path)
    logging.info('animes shape: %s', animes.shape)

    # Cargar dataset adicional con sinopsis/genres API
    logging.info('Cargando dataset_completo desde %s', dataset_path)
    ds = pd.read_csv(dataset_path)
    logging.info('dataset_completo shape: %s', ds.shape)

    # Intentar hacer merge por `animeID`; si no existe, intentamos por `title`
    if 'animeID' in animes.columns and 'animeID' in ds.columns:
        merged = pd.merge(animes, ds, on='animeID', how='outer', suffixes=('_anime','_ds'))
    else:
        merged = pd.merge(animes, ds, on='title', how='outer', suffixes=('_anime','_ds'))
    logging.info('Merged shape: %s', merged.shape)

    # Si existe ratings.csv lo procesamos en chunks
    if os.path.exists(ratings_path):
        logging.info('Procesando ratings desde %s (chunksize)...', ratings_path)
        # Intentar leer sólo el header para conocer columnas
        try:
            header = pd.read_csv(ratings_path, nrows=0)
            cols = list(header.columns)
            logging.info('Ratings columns detected: %s', cols)
        except Exception as e:
            logging.warning('No se pudo leer header de ratings.csv: %s', e)
            cols = None

        # En nuestros datasets la columna identificadora es `animeID`, la usamos directamente.
        # Para la columna de rating en `ratings.csv` esperamos el rating individual que da
        # un usuario ('rating'). Como fallback usamos 'score' si el CSV tuviera otro nombre.
        rating_col_candidates = ['rating', 'score']

        # Leer por chunks para ahorrar memoria
        reader = pd.read_csv(ratings_path, chunksize=200000)
        agg_list = []
        # Por cada chunk calculamos count, mean, std, median por anime
        for chunk in reader:
            if cols is None:
                cols = list(chunk.columns)
            # Preferimos la columna explícita 'animeID' en los chunks.
            if 'animeID' in chunk.columns:
                anime_col = 'animeID'
            else:
                # Fallback heurístico: tomar la segunda columna si no existe 'animeID'
                anime_col = chunk.columns[1]

            # Detectar columna con rating
            rating_col = None
            for c in rating_col_candidates:
                if c in chunk.columns:
                    rating_col = c
                    break
            if rating_col is None:
                # heuristic fallback: última columna
                rating_col = chunk.columns[-1]

            # Mantener solo las columnas relevantes y normalizar nombres
            local = chunk[[anime_col, rating_col]].copy()
            local.columns = ['anime_id', 'rating']
            # Forzar a numérico y convertir valores inválidos a NaN
            local['rating'] = pd.to_numeric(local['rating'], errors='coerce')
            # Agregados por anime
            g = local.groupby('anime_id')['rating'].agg(['count','mean','std','median']).rename(columns={'count':'rating_count','mean':'rating_mean','std':'rating_std','median':'rating_median'})
            agg_list.append(g)

        # Concatenar agregados y unir por anime_id
        if len(agg_list) > 0:
            agg_all = pd.concat(agg_list).groupby(level=0).agg({'rating_count':'sum','rating_mean':'mean','rating_std':'mean','rating_median':'mean'})
            agg_all.index.name = 'anime_id'
            agg_all = agg_all.reset_index()
            logging.info('Aggregados de ratings por anime: %s rows', agg_all.shape[0])
            # Merge con la tabla principal
            if 'animeID' in merged.columns:
                merged = pd.merge(merged, agg_all, left_on='animeID', right_on='anime_id', how='left')
            else:
                merged = pd.merge(merged, agg_all, left_on='animeID', right_on='anime_id', how='left')
        else:
            logging.warning('No se obtuvieron agregados de ratings (archivo vacío o formato inesperado).')
    else:
        logging.warning('ratings.csv no encontrado en ruta: %s. Se omiten características de usuarios.', ratings_path)

    return merged